# 🎯 Índice de Legitimidade (IL) — Detecção de Falso Favorito
### Custo/Valor do Gol · Luck Factor · Delta Modelo/Mercado · SOS · Z-Score

---

## Conceito

Baseado na lógica de **Custo e Valor do Gol** do Adriano Duarte (FullTrader), adaptada para jogos futuros com três melhorias:

1. **Gols ajustados por xG** — reduz ruído de conversão acima/abaixo do esperado  
2. **Peso amostral (N)** — quanto menor N, mais ancora no xG como referência  
3. **Índice composto (IL)** — combina 5 camadas independentes num score 0-1  

---

## Equações centrais

```
prob_mercado = 1 / odd_atual

confiança    = min(N_jogos / 10, 1.0)
alpha_efetivo = ALPHA × confiança
Gols_adj     = alpha_ef × Gols_N10 + (1 - alpha_ef) × xG_N10

Custo_Gol_Casa = Gols_adj_casa / prob_mercado_casa   [maior = melhor]
Valor_Gol_Casa = Gols_adj_casa × prob_mercado_fora   [maior = melhor]
Custo_Gol_Fora = Gols_adj_fora / prob_mercado_fora
Valor_Gol_Fora = Gols_adj_fora × prob_mercado_casa

Luck_Factor    = Gols_N10 - xG_N10   [próximo de 0 = conversão saudável]
Delta_Modelo   = prob_modelo - prob_mercado   [>0 = modelo vê mais valor]

IL = 0.25×EV + 0.20×SOS + 0.20×Z_sustentável + 0.20×Custo + 0.15×Luck
```

## Classificação final

| Classificação | Condição |
|---|---|
| `VALOR_REAL_CASA` | IL_casa > 0.65 + EV_bc > 3% |
| `FALSO_FAVORITO_CASA` | prob_mercado_casa > 55% + IL_casa < 35% |
| `POTENCIAL_CASA` | Delta > 8% + IL_casa > 50% |
| `NEUTRO` | Demais casos |


In [29]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 0 — Imports e Constantes
# ═══════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from IPython.display import display

# ── Caminhos ──────────────────────────────────────────────
PATH_INPUT  = 'rodada_atual_sosS_v3.csv'   # CSV com SOS + Z já calculados
PATH_OUTPUT = 'rodada_atual_sos_v3.csv'   # sobrescreve com novas colunas IL

# ── Parâmetros ────────────────────────────────────────────
ALPHA_SHRINKAGE = 0.60   # peso dos gols reais (0.6) vs xG âncora (0.4)
                          # 0.60: mantém 60% da realização real, 40% do xG
N_REF           = 10     # N de referência para confiança amostral
                          # N=10 → alpha_ef=0.60 | N=5 → alpha_ef=0.30
DELTA_THRESHOLD = 0.08   # divergência mínima modelo vs mercado (8%)

# Pesos do IL (somam 1.0)
PESOS_IL = {
    'ev':    0.25,   # EV positivo — mercado subestima o time
    'sos':   0.20,   # SOS alto — forma veio de tabela difícil
    'z':     0.20,   # Z sustentável — não está em pico atípico
    'custo': 0.20,   # Custo do Gol alto — eficiência real contra expectativa
    'luck':  0.15,   # Luck Factor baixo — não depende de sorte na conversão
}

print('✅ Constantes configuradas')
print(f'   ALPHA_SHRINKAGE = {ALPHA_SHRINKAGE}')
print(f'   N_REF           = {N_REF}')
print(f'   DELTA_THRESHOLD = {DELTA_THRESHOLD}')
print(f'   Pesos IL: {PESOS_IL}')

✅ Constantes configuradas
   ALPHA_SHRINKAGE = 0.6
   N_REF           = 10
   DELTA_THRESHOLD = 0.08
   Pesos IL: {'ev': 0.25, 'sos': 0.2, 'z': 0.2, 'custo': 0.2, 'luck': 0.15}


In [30]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 1 — Carregar CSV
# ═══════════════════════════════════════════════════════════
ra = pd.read_csv(PATH_INPUT)

print(f'rodada_atual: {ra.shape[0]} jogos | {ra.shape[1]} colunas')

# Verificar colunas obrigatórias
obrigatorias = [
    'gols_marc_home', 'gols_marc_away',
    'xg_home_n10', 'xg_away_n10',
    'n10_home', 'n10_away',
    'odd_home_d2', 'odd_away_d2',
    'prob_bc', 'prob_bf',
    'ev_bc', 'ev_bf',
    'sos_home_norm', 'sos_away_norm',
    'z_gols_home_adj', 'z_gols_away_adj',
]
faltando = [c for c in obrigatorias if c not in ra.columns]
if faltando:
    raise ValueError(f'⚠️  Colunas ausentes: {faltando}\n'
                     f'   Rode calcular_sos.ipynb antes deste notebook.')
print('✅ Todas as colunas obrigatórias presentes')
display(ra[['league','home_team','away_team',
            'gols_marc_home','xg_home_n10','n10_home',
            'odd_home_d2','odd_away_d2']].head(5))

rodada_atual: 26 jogos | 80 colunas
✅ Todas as colunas obrigatórias presentes


,league,home_team,away_team,gols_marc_home,xg_home_n10,n10_home,odd_home_d2,odd_away_d2
0,Espanha_2 Segunda Division,Albacete Balompié,CD Castellón,1.2,1.411,10,3.44,2.05
1,Espanha_2 Segunda Division,Almería,Real Sociedad II,1.8,1.516,10,1.56,5.40
2,Espanha_2 Segunda Division,Real Valladolid,Burgos CF,1.2,1.307,10,2.25,3.38
3,Espanha_2 Segunda Division,Ceuta,Cádiz,1.3,1.339,10,2.18,3.43
4,Espanha_2 Segunda Division,Málaga CF,Leganés,2.2,1.715,10,2.05,3.73


In [31]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 2 — Funções de cálculo
# ═══════════════════════════════════════════════════════════

def odd_para_prob(serie_odd):
    """
    Converte odd decimal em probabilidade implícita.
    prob = 1 / odd
    Valores inválidos (<=1.0 ou NaN) retornam NaN.
    """
    o = pd.to_numeric(serie_odd, errors='coerce')
    return np.where(o > 1.0, 1.0 / o, np.nan)


def calcular_gols_ajustados(gols, xg, n, alpha=ALPHA_SHRINKAGE, n_ref=N_REF):
    """
    Gols ajustado pelo xG com peso amostral.

    Lógica:
      - Quando N = n_ref (amostra completa): alpha_ef = alpha (ex: 0.60)
        → 60% gols reais + 40% xG
      - Quando N < n_ref (amostra pequena): alpha_ef diminui
        → mais peso no xG como âncora estável
      - Quando N = 0: alpha_ef = 0 → usa só xG

    Objetivo: eliminar dois ruídos simultaneamente
      1. Conversão acima/abaixo do xG (Luck Factor)
      2. Variância amostral de N pequeno
    """
    confianca  = np.minimum(n / n_ref, 1.0)
    alpha_ef   = alpha * confianca
    return alpha_ef * gols + (1.0 - alpha_ef) * xg


def normalizar_01(serie, q_low=0.05, q_high=0.95):
    """
    Normaliza serie para [0,1] usando percentis para robustez.
    Clip garante que outliers não distorcem o range.
    """
    lo = serie.quantile(q_low)
    hi = serie.quantile(q_high)
    if hi == lo:
        return pd.Series(0.5, index=serie.index)
    return ((serie - lo) / (hi - lo)).clip(0, 1)


print('✅ Funções definidas')

# Teste rápido
_g  = calcular_gols_ajustados(3.0, 2.156, 10)
_gp = calcular_gols_ajustados(3.0, 2.156, 5)
print(f'   Teste N=10: gols_adj={_g:.3f}  (alpha_ef={0.6*min(10/10,1):.2f})')
print(f'   Teste N=5 : gols_adj={_gp:.3f}  (alpha_ef={0.6*min(5/10,1):.2f})')

✅ Funções definidas
   Teste N=10: gols_adj=2.662  (alpha_ef=0.60)
   Teste N=5 : gols_adj=2.409  (alpha_ef=0.30)


In [32]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 3 — Custo e Valor do Gol
# ═══════════════════════════════════════════════════════════

# ── Odds: preferir atual, fallback para D-2 ──────────────
odd_home = ra['odd_home_atual'].where(
    pd.to_numeric(ra['odd_home_atual'], errors='coerce') > 1.0,
    ra['odd_home_d2']
)
odd_away = ra['odd_away_atual'].where(
    pd.to_numeric(ra['odd_away_atual'], errors='coerce') > 1.0,
    ra['odd_away_d2']
)

prob_casa = odd_para_prob(odd_home)
prob_fora = odd_para_prob(odd_away)

ra['prob_mercado_casa'] = prob_casa.round(4)
ra['prob_mercado_fora'] = prob_fora.round(4)

# ── Gols ajustados ───────────────────────────────────────
gols_h_adj = calcular_gols_ajustados(
    ra['gols_marc_home'], ra['xg_home_n10'], ra['n10_home']
)
gols_a_adj = calcular_gols_ajustados(
    ra['gols_marc_away'], ra['xg_away_n10'], ra['n10_away']
)
ra['gols_home_adj'] = gols_h_adj.round(3)
ra['gols_away_adj'] = gols_a_adj.round(3)

# ── Custo do Gol ─────────────────────────────────────────
# Adriano: Gols / prob_própria
# Adaptado: Gols_adj / prob_mercado_própria
# Maior = mais eficiente sendo underdog (ou neutro)
ra['custo_gol_casa'] = (gols_h_adj / prob_casa).round(4)
ra['custo_gol_fora'] = (gols_a_adj / prob_fora).round(4)

# ── Valor do Gol ─────────────────────────────────────────
# Adriano: Gols × prob_adversário
# Adaptado: Gols_adj × prob_mercado_adversário
# Maior = marca contra times que o mercado considera fortes
ra['valor_gol_casa'] = (gols_h_adj * prob_fora).round(4)
ra['valor_gol_fora'] = (gols_a_adj * prob_casa).round(4)

# ── Luck Factor ──────────────────────────────────────────
# ΔxG = Gols_reais - xG_esperado
# +: converte acima do xG (pode ser habilidade ou sorte)
# -: converte abaixo do xG (azar ou desperdício)
# Próximo de 0 = conversão saudável e sustentável
ra['luck_home'] = (ra['gols_marc_home'] - ra['xg_home_n10']).round(3)
ra['luck_away'] = (ra['gols_marc_away'] - ra['xg_away_n10']).round(3)

# ── Delta Modelo vs Mercado ───────────────────────────────
# Delta > 0: modelo acha o time mais forte do que a odd sugere
# Delta < 0: mercado precifica mais otimismo do que os dados justificam
ra['delta_bc'] = (ra['prob_bc'] - prob_casa).round(4)
ra['delta_bf'] = (ra['prob_bf'] - prob_fora).round(4)

print('✅ Custo/Valor do Gol calculados')
print()
display(ra[['home_team','away_team',
            'gols_marc_home','gols_home_adj','xg_home_n10',
            'custo_gol_casa','valor_gol_casa',
            'luck_home','delta_bc']].head(8))

✅ Custo/Valor do Gol calculados



,home_team,away_team,gols_marc_home,gols_home_adj,xg_home_n10,custo_gol_casa,valor_gol_casa,luck_home,delta_bc
0,Albacete Balompié,CD Castellón,1.2,1.284,1.411,4.4183,0.6265,-0.211,0.0426
1,Almería,Real Sociedad II,1.8,1.686,1.516,2.6308,0.3123,0.284,-0.1883
2,Real Valladolid,Burgos CF,1.2,1.243,1.307,2.7963,0.3677,-0.107,-0.0846
3,Ceuta,Cádiz,1.3,1.316,1.339,2.8680,0.3836,-0.039,0.0268
4,Málaga CF,Leganés,2.2,2.006,1.715,4.1123,0.5378,0.485,0.0942
5,SD Eibar,UD Las Palmas,1.2,1.244,1.311,3.0488,0.3715,-0.111,0.0511
6,Cultural y Deportiva Leonesa,FC Andorra,0.8,1.011,1.327,2.4259,0.3369,-0.527,-0.1059
7,Emmen,Cambuur,0.9,1.098,1.396,3.1854,0.5438,-0.496,-0.1470


In [33]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 4 — Índice de Legitimidade (IL)
# ═══════════════════════════════════════════════════════════
#
# 5 camadas independentes, cada uma normalizada em [0,1]
# Cada camada elimina um tipo de ruído diferente
#
#  Camada 1 — EV (25%): mercado subestima o time?
#  Camada 2 — SOS (20%): forma veio de tabela difícil?
#  Camada 3 — Z (20%): nível de gols é sustentável?
#  Camada 4 — Custo do Gol (20%): eficiência real contra expectativa?
#  Camada 5 — Luck Factor (15%): não depende de sorte na conversão?

# ── Componentes para casa ─────────────────────────────────
ev_bc_fill   = ra['ev_bc'].fillna(0)
c1_ev_h      = normalizar_01(ev_bc_fill)                      # EV positivo
c2_sos_h     = normalizar_01(ra['sos_home_norm'])              # SOS alto
c3_z_h       = normalizar_01(-ra['z_gols_home_adj'].abs())    # Z próximo de 0
c4_custo_h   = normalizar_01(ra['custo_gol_casa'])             # Custo alto
c5_luck_h    = normalizar_01(-ra['luck_home'].abs())           # Luck próximo de 0

ra['IL_casa'] = (
    PESOS_IL['ev']    * c1_ev_h +
    PESOS_IL['sos']   * c2_sos_h +
    PESOS_IL['z']     * c3_z_h +
    PESOS_IL['custo'] * c4_custo_h +
    PESOS_IL['luck']  * c5_luck_h
).round(4)

# ── Componentes para fora ─────────────────────────────────
ev_bf_fill   = ra['ev_bf'].fillna(0)
c1_ev_a      = normalizar_01(ev_bf_fill)
c2_sos_a     = normalizar_01(ra['sos_away_norm'])
c3_z_a       = normalizar_01(-ra['z_gols_away_adj'].abs())
c4_custo_a   = normalizar_01(ra['custo_gol_fora'])
c5_luck_a    = normalizar_01(-ra['luck_away'].abs())

ra['IL_fora'] = (
    PESOS_IL['ev']    * c1_ev_a +
    PESOS_IL['sos']   * c2_sos_a +
    PESOS_IL['z']     * c3_z_a +
    PESOS_IL['custo'] * c4_custo_a +
    PESOS_IL['luck']  * c5_luck_a
).round(4)

print('✅ IL calculado')
print(f'   IL_casa: min={ra["IL_casa"].min():.3f} max={ra["IL_casa"].max():.3f} μ={ra["IL_casa"].mean():.3f}')
print(f'   IL_fora: min={ra["IL_fora"].min():.3f} max={ra["IL_fora"].max():.3f} μ={ra["IL_fora"].mean():.3f}')

# Distribuição
bins = [0, 0.25, 0.40, 0.55, 0.70, 1.0]
labs = ['Muito baixo','Baixo','Médio','Alto','Muito alto']
print('\nDistribuição IL casa:')
print(pd.cut(ra['IL_casa'], bins=bins, labels=labs).value_counts().sort_index())

✅ IL calculado
   IL_casa: min=0.220 max=0.857 μ=0.480
   IL_fora: min=0.305 max=0.799 μ=0.542

Distribuição IL casa:
IL_casa
Muito baixo    1
Baixo          3
Médio          6
Alto           3
Muito alto     1
Name: count, dtype: int64


In [34]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 5 — Classificação: Falso Favorito vs Valor Real
# ═══════════════════════════════════════════════════════════

def classificar_jogo(row):
    """
    Classifica cada jogo com base no IL e nos sinais de mercado.

    Hierarquia de prioridade:
    1. FALSO_FAVORITO: mercado favorece fortemente + IL baixo
       → mercado erra na direção perigosa para quem aposta no favorito
    2. VALOR_REAL: IL alto + EV positivo
       → todos os sinais alinham na mesma direção
    3. POTENCIAL: Delta alto + IL médio
       → modelo enxerga valor mas sem confirmação total
    4. NEUTRO: sem sinal claro
    """
    il_h  = row['IL_casa']
    il_a  = row['IL_fora']
    pm_h  = row['prob_mercado_casa'] or 0
    pm_a  = row['prob_mercado_fora'] or 0
    ev_h  = row['ev_bc']  if pd.notna(row['ev_bc'])  else 0
    ev_a  = row['ev_bf']  if pd.notna(row['ev_bf'])  else 0
    dlt_h = row['delta_bc'] or 0
    dlt_a = row['delta_bf'] or 0

    # Falso favorito: mercado muito otimista + IL não sustenta
    if pm_h > 0.55 and il_h < 0.35:
        return 'FALSO_FAVORITO_CASA'
    if pm_a > 0.55 and il_a < 0.35:
        return 'FALSO_FAVORITO_FORA'

    # Valor real: IL alto + EV positivo (todos os sinais alinhados)
    if il_h > 0.65 and ev_h > 0.03:
        return 'VALOR_REAL_CASA'
    if il_a > 0.65 and ev_a > 0.03:
        return 'VALOR_REAL_FORA'

    # Potencial: delta forte + IL médio (modelo vê valor, dados parcialmente confirmam)
    if dlt_h > DELTA_THRESHOLD and il_h > 0.50:
        return 'POTENCIAL_CASA'
    if dlt_a > DELTA_THRESHOLD and il_a > 0.50:
        return 'POTENCIAL_FORA'

    return 'NEUTRO'

ra['classificacao_IL'] = ra.apply(classificar_jogo, axis=1)

print('Distribuição das classificações:')
print(ra['classificacao_IL'].value_counts())

Distribuição das classificações:
classificacao_IL
NEUTRO                 19
VALOR_REAL_FORA         4
VALOR_REAL_CASA         2
FALSO_FAVORITO_CASA     1
Name: count, dtype: int64


In [35]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 6 — Análise por classificação
# ═══════════════════════════════════════════════════════════

cols_view = ['league','home_team','away_team',
             'IL_casa','IL_fora','custo_gol_casa','valor_gol_casa',
             'luck_home','delta_bc','sos_home_norm','z_gols_home_adj',
             'ev_bc','prob_bc','prob_mercado_casa']

print('=== VALOR REAL CASA ===')
vrc = ra[ra['classificacao_IL']=='VALOR_REAL_CASA'].sort_values('IL_casa', ascending=False)
display(vrc[cols_view])

print('\n=== VALOR REAL FORA ===')
vrf = ra[ra['classificacao_IL']=='VALOR_REAL_FORA'].sort_values('IL_fora', ascending=False)
cols_fora = ['league','home_team','away_team',
             'IL_fora','custo_gol_fora','valor_gol_fora',
             'luck_away','delta_bf','sos_away_norm','z_gols_away_adj',
             'ev_bf','prob_bf','prob_mercado_fora']
display(vrf[cols_fora])

print('\n=== FALSO FAVORITO CASA ===')
ffc = ra[ra['classificacao_IL']=='FALSO_FAVORITO_CASA'].sort_values('IL_casa')
display(ffc[cols_view])

print('\n=== FALSO FAVORITO FORA ===')
fff = ra[ra['classificacao_IL']=='FALSO_FAVORITO_FORA'].sort_values('IL_fora')
display(fff[cols_fora])

print('\n=== POTENCIAL CASA/FORA ===')
pot = ra[ra['classificacao_IL'].str.startswith('POTENCIAL')]
display(pot[cols_view + ['classificacao_IL']])

=== VALOR REAL CASA ===


,league,home_team,away_team,IL_casa,IL_fora,custo_gol_casa,valor_gol_casa,luck_home,delta_bc,sos_home_norm,z_gols_home_adj,ev_bc,prob_bc,prob_mercado_casa
0,Espanha_2 Segunda Division,Albacete Balompié,CD Castellón,0.8567,0.3052,4.4183,0.6265,-0.211,0.0426,1.0119,-0.4913,0.146615,0.333318,0.2907
5,Espanha_2 Segunda Division,SD Eibar,UD Las Palmas,0.6799,0.5682,3.0488,0.3715,-0.111,0.0511,0.9981,-0.4980,0.125247,0.459284,0.4082



=== VALOR REAL FORA ===


,league,home_team,away_team,IL_fora,custo_gol_fora,valor_gol_fora,luck_away,delta_bf,sos_away_norm,z_gols_away_adj,ev_bf,prob_bf,prob_mercado_fora
17,Colombia_1 Liga BetPlay,América de Cali,Llaneros,0.7991,6.7729,0.7003,0.019,0.0544,0.9981,-0.2865,0.337284,0.215691,0.1613
19,Colombia_1 Liga BetPlay,Junior,Atlético Bucaramanga,0.7449,4.7026,0.6192,0.141,0.0853,1.0079,0.2303,0.298715,0.371061,0.2857
21,Colombia_1 Liga BetPlay,Fortaleza CEIF,Deportivo Pasto,0.6927,4.6773,0.6796,-0.022,0.1054,0.9993,0.4051,0.326813,0.428004,0.3226
6,Espanha_2 Segunda Division,Cultural y Deportiva Leonesa,FC Andorra,0.6525,4.5228,0.6282,-0.269,0.0636,1.0085,0.2301,0.190903,0.396968,0.3333



=== FALSO FAVORITO CASA ===


,league,home_team,away_team,IL_casa,IL_fora,custo_gol_casa,valor_gol_casa,luck_home,delta_bc,sos_home_norm,z_gols_home_adj,ev_bc,prob_bc,prob_mercado_casa
1,Espanha_2 Segunda Division,Almería,Real Sociedad II,0.2871,0.65,2.6308,0.3123,0.284,-0.1883,0.9745,0.4074,-0.293785,0.452702,0.641



=== FALSO FAVORITO FORA ===


,league,home_team,away_team,IL_fora,custo_gol_fora,valor_gol_fora,luck_away,delta_bf,sos_away_norm,z_gols_away_adj,ev_bf,prob_bf,prob_mercado_fora



=== POTENCIAL CASA/FORA ===


,league,home_team,away_team,IL_casa,IL_fora,custo_gol_casa,valor_gol_casa,luck_home,delta_bc,sos_home_norm,z_gols_home_adj,ev_bc,prob_bc,prob_mercado_casa,classificacao_IL


In [36]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 7 — Exportar CSV final
# ═══════════════════════════════════════════════════════════

novas_colunas = [
    'prob_mercado_casa', 'prob_mercado_fora',
    'gols_home_adj',     'gols_away_adj',
    'custo_gol_casa',    'custo_gol_fora',
    'valor_gol_casa',    'valor_gol_fora',
    'luck_home',         'luck_away',
    'delta_bc',          'delta_bf',
    'IL_casa',           'IL_fora',
    'classificacao_IL',
]

ra.to_csv(PATH_OUTPUT, index=False)

print(f'✅ Exportado: {PATH_OUTPUT}')
print(f'   Shape final: {ra.shape}')
print()
print('Novas colunas adicionadas:')
for c in novas_colunas:
    presente = '✓' if c in ra.columns else '✗'
    print(f'  {presente} {c}')

print()
print('📌 Próximo passo: enviar rodada_atual_sos_v3.csv para o Claude')
print('   gerar index.html com IL embutido no painel.')
display(ra[['home_team','away_team','IL_casa','IL_fora','classificacao_IL']].head(10))

✅ Exportado: rodada_atual_sos_v3.csv
   Shape final: (26, 95)

Novas colunas adicionadas:
  ✓ prob_mercado_casa
  ✓ prob_mercado_fora
  ✓ gols_home_adj
  ✓ gols_away_adj
  ✓ custo_gol_casa
  ✓ custo_gol_fora
  ✓ valor_gol_casa
  ✓ valor_gol_fora
  ✓ luck_home
  ✓ luck_away
  ✓ delta_bc
  ✓ delta_bf
  ✓ IL_casa
  ✓ IL_fora
  ✓ classificacao_IL

📌 Próximo passo: enviar rodada_atual_sos_v3.csv para o Claude
   gerar index.html com IL embutido no painel.


,home_team,away_team,IL_casa,IL_fora,classificacao_IL
0,Albacete Balompié,CD Castellón,0.8567,0.3052,VALOR_REAL_CASA
1,Almería,Real Sociedad II,0.2871,0.6500,FALSO_FAVORITO_CASA
2,Real Valladolid,Burgos CF,0.4617,0.5785,NEUTRO
3,Ceuta,Cádiz,0.6486,0.3396,NEUTRO
4,Málaga CF,Leganés,0.4611,0.5282,NEUTRO
5,SD Eibar,UD Las Palmas,0.6799,0.5682,VALOR_REAL_CASA
6,Cultural y Deportiva Leonesa,FC Andorra,0.2885,0.6525,VALOR_REAL_FORA
7,Emmen,Cambuur,0.2204,0.3484,NEUTRO
8,Eindhoven,Emmen,NaN,NaN,NEUTRO
9,Willem II,De Graafschap,NaN,NaN,NEUTRO
